# MiniMind-O 03 — Colab WebUI 启动器 (Gradio & 实时语音通话)

本 Notebook 用于在 Google Colab 上一键部署两种 WebUI：
- **Gradio WebUI (`scripts/web_demo_omni.py`)**：通过 `share=True` 获取 `https://*.gradio.live` 公网链接，支持文本、图像、语音上传与麦克风输入。
- **Realtime Phone-Mode WebUI (`webui/web_demo.py`)**：基于 Flask + WebSocket + Silero VAD，通过 `cloudflared` 隧道提供 `https://*.trycloudflare.com`，支持流式语音合成与实时打断（barge-in）。

In [ ]:
#@title 1. 环境初始化与依赖安装 (Colab / 本地 Jupyter 通用)
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/jingyaogong/minimind-o.git"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"} # 留空克隆默认分支，或填入特定分支名
INSTALL_DEPS = True  #@param {type:"boolean"}

def _find_repo_root():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/minimind-o"]:
        if os.path.exists(os.path.join(cand, "notebook_utils.py")) and os.path.exists(os.path.join(cand, "model", "model_omni.py")):
            return os.path.abspath(cand)
    return None

root = _find_repo_root()
if root is None:
    target = "/content/minimind-o" if "google.colab" in sys.modules else os.path.abspath("./minimind-o")
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        cmd += ["-b", BRANCH.strip()]
    cmd += [REPO_URL, target]
    print("Cloning repository:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    root = target

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

if INSTALL_DEPS:
    req_file = "requirements_colab.txt" if os.path.exists("requirements_colab.txt") else "requirements.txt"
    print(f"Installing dependencies from {req_file} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)
    if shutil.which("ffmpeg") is None and "google.colab" in sys.modules:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

from notebook_utils import check_environment
env_info = check_environment()

In [ ]:
#@title 2. 下载编码器与 Transformers 格式模型
SOURCE = "huggingface"  #@param ["huggingface", "modelscope"]
MODEL_VARIANT = "minimind-3o"  #@param ["minimind-3o", "minimind-3o-moe"]

from notebook_utils import download_encoders, download_hf_model

download_encoders(source=SOURCE, include_campplus=True)
download_hf_model(model_name=MODEL_VARIANT, source=SOURCE, target_dir=os.path.join(".", MODEL_VARIANT))

In [ ]:
#@title 3. 选择并启动 WebUI
WEBUI_TYPE = "gradio"  #@param ["gradio", "realtime_flask_cloudflared"]
MODELS_SCAN_DIR = "./"  #@param {type:"string"}
PORT = 7860  #@param {type:"integer"}
OPEN_THINKING = False  #@param {type:"boolean"}
MAX_HISTORY_TURNS = 2  #@param {type:"integer"}

if WEBUI_TYPE == "gradio":
    from scripts.web_demo_omni import start_gradio_demo
    start_gradio_demo(
        load_from=MODELS_SCAN_DIR,
        open_thinking=int(OPEN_THINKING),
        port=PORT,
        share=True,
    )
else:
    from webui.web_demo import start_realtime_webui
    from notebook_utils import launch_cloudflared_tunnel
    start_realtime_webui(
        load_from=MODELS_SCAN_DIR,
        port=PORT,
        max_history_turns=MAX_HISTORY_TURNS,
        background=True,
    )
    tunnel_proc, public_url = launch_cloudflared_tunnel(port=PORT)